In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, re, json, string, glob
import numpy as np, pandas as pd
pd.set_option("display.width", 200)
RULE_VER = "zone_v1"      # 결과 파일에 박아둘 규칙 버전

Mounted at /content/drive


In [ ]:
P9  = "/content/drive/MyDrive/Grad_share/data_share/P9"
OUT = f"{P9}/SCORE"; os.makedirs(OUT, exist_ok=True)

for d in [P9, f"{P9}/T456"]:
    print(f"\n=== {d} ===")
    for f in sorted(glob.glob(f"{d}/*")):
        print(f"  {os.path.basename(f):50} {os.path.getsize(f)/1e6:8.2f} MB")


=== /content/drive/MyDrive/Grad_share/data_share/P9 ===
  A1_generations.parquet                                 0.32 MB
  A1_saturation.csv                                      0.00 MB
  A2_p8s_cell_summary.csv                                0.00 MB
  A2_p8s_diag.csv                                        0.23 MB
  A2_purity_by_dataset.csv                               0.00 MB
  A2_sv_sf_labels.csv                                    0.07 MB
  B2_generations.parquet                                 0.17 MB
  B2_rerun_generations.parquet                           0.15 MB
  B2_results.csv                                         0.00 MB
  B3_generations.parquet                                 0.65 MB
  B3_rerun_generations.parquet                           0.12 MB
  B3_rerun_results.csv                                   0.00 MB
  B3_results.csv                                         0.00 MB
  B3_unconditional.csv                                   0.00 MB
  ENV_fingerprint_0830.csv       

In [ ]:
pairs = pd.read_csv(f"{P9}/p8s_behavior.csv")
print("pairs:", pairs.shape)

T456 = f"{P9}/T456"
GEN_FILES = [f"{T456}/T4_generations.parquet",
             f"{T456}/T5_generations.parquet",
             f"{T456}/T5_valid_generations.parquet",
             f"{T456}/T5a_generations.parquet"]
G = pd.concat([pd.read_parquet(f).assign(src=os.path.basename(f)[:-8]) for f in GEN_FILES],
              ignore_index=True)
print("\nG:", G.shape, "| 컬럼:", list(G.columns))
print(G.groupby(["src","cond"]).size().to_string())

for c in ["row","cond","gen"]:
    assert c in G.columns, f"★ 누락: {c}"
assert G.row.isin(pairs.index).all(), "★ row 인덱스 불일치"

# ── 기존 수동 채점본 (규칙 검증용 정답지) ──
KEY = pd.read_csv(f"{T456}/semantic_audit_key.csv")
print("\nsemantic_audit_key:", KEY.shape, "| 컬럼:", list(KEY.columns))
print(KEY.head(3).to_string())
with open(f"{T456}/verdicts.json") as f: V = json.load(f)
print("\nverdicts.json 타입:", type(V), "| 길이:", len(V))
print(str(V)[:400])

pairs: (1224, 19)

G: (13788, 5) | 컬럼: ['cond', 'seed', 'row', 'gen', 'src']
src                   cond         
T4_generations        KmCm_base         234
                      KmCm_c2           234
                      KmCm_c3           234
                      KmCm_c5           234
                      KpCp_base         378
                      KpCp_c-3          378
                      KpCp_c-5          378
T5_generations        front_all        1134
                      front_last       1134
                      signal_all       1134
                      signal_last      1134
T5_valid_generations  gold_v1          1134
                      gold_v1_front    1134
                      random           1134
T5a_generations       baseline          378
                      both_accept      1134
                      both_ret_n63     1134
                      both_retain      1134

semantic_audit_key: (241, 8) | 컬럼: ['case_id', 'cond', 'tag', 'row', 'question', 'gold', 'ctx_

In [ ]:
# case_id가 (row, cond, seed)로 되돌려지는지 확인 — 안 되면 규칙 검증 불가
print("KEY 고유 케이스:", len(KEY))
for c in ["case_id","row","cond","seed","verdict","v3","gen"]:
    print(f"  {c:10}", "있음" if c in KEY.columns else "없음")
print("\nKEY 컬럼 샘플:")
print(KEY.iloc[:2].T.to_string())

KEY 고유 케이스: 241
  case_id    있음
  row        있음
  cond       있음
  seed       없음
  verdict    없음
  v3         없음
  gen        없음

KEY 컬럼 샘플:
                                                                                                                                                                                                       0                                                                                                                                                                                                 1
case_id                                                                                                                                                                                                0                                                                                                                                                                                                 1
cond                                                                  

In [ ]:
def normalize(s):
    s = str(s).lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    return " ".join(s.split())

def as_list(g):
    return json.loads(g) if str(g).strip().startswith("[") else [g]

def spans(ntext, alias_list):
    out = []
    for a in alias_list:
        na = normalize(a)
        if not na: continue
        out += [m.span() for m in re.finditer(r"\b"+re.escape(na)+r"\b", ntext)]
    return out

NEG = re.compile(r"\b(not|no|never|isnt|arent|wasnt|doesnt|dont|didnt|cannot|"
                 r"rather than|instead of|incorrect|wrong|mistake|false|contrary to)\b")
WIN = 45

def negated(nt, sp): return bool(NEG.search(nt[max(0,sp[0]-WIN):sp[0]]))

def zone_of(gen, golds, ctxs, overlap=False):
    t = str(gen); nt = normalize(t)
    gs, cs = spans(nt, golds), spans(nt, ctxs)
    g, c = len(gs)>0, len(cs)>0
    trunc = not re.search(r"[.!?]\s*$", t.strip())
    if overlap:                              return "JUDGE","overlap",trunc
    if g and c:                              return "JUDGE","both",trunc
    if g and all(negated(nt,s) for s in gs): return "JUDGE","neg_gold",trunc
    if c and all(negated(nt,s) for s in cs): return "JUDGE","neg_ctx",trunc
    if g: return "GOLD","gold_only",trunc
    if c: return "CTX","ctx_only",trunc
    return "NONE","neither",trunc

In [ ]:
GOLDS = {i: as_list(pairs.loc[i,"gold"]) for i in pairs.index}
CTXS  = {i: [pairs.loc[i,"answer_surface"]] for i in pairs.index}

OVL = set()
for i in pairs.index:
    ng = [normalize(x) for x in GOLDS[i] if normalize(x)]
    nc = normalize(CTXS[i][0])
    if nc and any(nc in x or x in nc for x in ng): OVL.add(i)
print(f"gold⊂ctx 포함관계 문항: {len(OVL)}개\n")

z = [zone_of(r.gen, GOLDS[r.row], CTXS[r.row], r.row in OVL) for r in G.itertuples()]
G["zone"], G["reason"], G["trunc"] = zip(*z)
G["rule_ver"] = RULE_VER

print("=== 전체 구역 분포 ==="); print(G.zone.value_counts(normalize=True).round(3).to_string())
print("\n=== 사유별 ==="); print(G.reason.value_counts().to_string())
print("\n=== 조건별 (T5 계열만) ===")
T5 = G[G.src.str.startswith("T5")]
print(T5.groupby("cond").agg(n=("zone","size"),
        JUDGE=("zone",lambda s:(s=="JUDGE").mean()),
        GOLD =("zone",lambda s:(s=="GOLD").mean()),
        trunc=("trunc","mean")).round(3).to_string())
print(f"\n→ JUDGE 총 {(G.zone=='JUDGE').sum()}건 / {len(G)}건")
G.to_parquet(f"{OUT}/zones_{RULE_VER}.parquet", index=False)

gold⊂ctx 포함관계 문항: 605개

=== 전체 구역 분포 ===
zone
JUDGE    0.330
CTX      0.312
GOLD     0.221
NONE     0.137

=== 사유별 ===
reason
ctx_only     4301
gold_only    3049
both         2951
neither      1893
overlap      1225
neg_ctx       310
neg_gold       59

=== 조건별 (T5 계열만) ===
                  n  JUDGE   GOLD  trunc
cond                                    
baseline        378  0.214  0.272  0.471
both_accept    1134  0.282  0.269  0.560
both_ret_n63   1134  0.343  0.229  0.558
both_retain    1134  0.349  0.233  0.573
front_all      1134  0.281  0.276  0.480
front_last     1134  0.263  0.261  0.490
gold_v1        1134  0.258  0.273  0.587
gold_v1_front  1134  0.220  0.280  0.510
random         1134  0.220  0.255  0.487
signal_all     1134  0.349  0.233  0.573
signal_last    1134  0.322  0.245  0.579

→ JUDGE 총 4545건 / 13788건


In [ ]:
M7to3 = {"gold_claim":"GOLD","both_gold":"GOLD","both_true":"GOLD",
         "ctx_claim":"CTX","both_ctx":"CTX","abstain":"NONE","unclear":"NONE"}

K = KEY.copy()
K["human7"] = K.case_id.astype(str).map(V)
K = K[K.human7.notna()].copy()
K["human"] = K.human7.map(M7to3)
K[["zone","reason","trunc"]] = pd.DataFrame(
    [zone_of(r.output, as_list(r.gold), [r.ctx_answer], r.row in OVL) for r in K.itertuples()],
    index=K.index)

print(f"수동 채점 {len(K)}건 | 사람 7범주:", K.human7.value_counts().to_dict())
print("\n=== 규칙 zone × 사람 3범주 ===")
print(pd.crosstab(K.zone, K.human, margins=True).to_string())

auto = K[K.zone!="JUDGE"]
print(f"\n=== 자동 확정 구역 {len(auto)}건 정확도 ===")
print(f"  일치 {(auto.zone==auto.human).sum()}/{len(auto)} = {(auto.zone==auto.human).mean():.3f}")
if (auto.zone!=auto.human).any():
    print("\n--- 불일치 사례 ---")
    for r in auto[auto.zone!=auto.human].itertuples():
        print(f"  [rule {r.zone}/{r.reason} vs human {r.human7}] gold={as_list(r.gold)[:2]} ctx={r.ctx_answer}")
        print(f"    {str(r.output)[:180]}\n")
print(f"JUDGE로 넘어간 비율: {(K.zone=='JUDGE').mean():.3f} (어려운 표본이라 높게 나오는 게 정상)")

수동 채점 60건 | 사람 7범주: {'both_gold': 28, 'gold_claim': 21, 'both_true': 6, 'unclear': 3, 'both_ctx': 1, 'ctx_claim': 1}

=== 규칙 zone × 사람 3범주 ===
human  CTX  GOLD  NONE  All
zone                       
CTX      0     0     1    1
GOLD     0    23     1   24
JUDGE    2    32     1   35
All      2    55     3   60

=== 자동 확정 구역 25건 정확도 ===
  일치 23/25 = 0.920

--- 불일치 사례 ---
  [rule GOLD/gold_only vs human unclear] gold=['association football', 'football'] ctx=cross-country skiing
    The text doesn't mention a sport at all. It appears to be a description of a Finnish football (soccer) league, but it doesn't explicitly state the sport. However, based on the cont

  [rule CTX/ctx_only vs human unclear] gold=['Lisbon', 'Lisboa'] ctx=Sri Jayawardenepura Kotte
    The question is incomplete, but based on the text, I can provide the answer.

The text mentions "Sri Jayawardenepura Kotte" as the country's capital, but this is actually the capit

JUDGE로 넘어간 비율: 0.583 (어려운 표본이라 높게 나오는 게 정상)


In [ ]:
## Cell 5 — overlap 판정을 단어 경계 기준으로 교체
def contains_wb(hay, needle):
    return bool(needle) and bool(re.search(r"\b"+re.escape(needle)+r"\b", hay))

OVL2 = set()
for i in pairs.index:
    ng = [normalize(x) for x in GOLDS[i] if normalize(x)]
    nc = normalize(CTXS[i][0])
    if nc and any(contains_wb(nc,x) or contains_wb(x,nc) for x in ng): OVL2.add(i)
print(f"overlap: {len(OVL)} → {len(OVL2)}개")
print("빠진 예시:", [(GOLDS[i][:2], CTXS[i][0]) for i in list(OVL-OVL2)[:5]])

z = [zone_of(r.gen, GOLDS[r.row], CTXS[r.row], r.row in OVL2) for r in G.itertuples()]
G["zone"], G["reason"], G["trunc"] = zip(*z)
print("\n=== 재분할 ==="); print(G.zone.value_counts(normalize=True).round(3).to_string())
print(G.reason.value_counts().to_string())

print("\n=== 절단 여부별 구역 (절단이 NONE을 만들어내는가) ===")
print(pd.crosstab(G.trunc, G.zone, normalize="index").round(3).to_string())

overlap: 605 → 601개
빠진 예시: [(['Japan', 'State of Japan'], 'Azerbaijan'), (['Canada', 'Dominion of Canada'], 'South Africa'), (['United States of America', 'the United States of America'], 'Austria'), (['Los Angeles', 'Los Angeles, California'], 'Holyoke')]

=== 재분할 ===
zone
JUDGE    0.326
CTX      0.312
GOLD     0.224
NONE     0.137
reason
ctx_only     4307
gold_only    3092
both         2999
neither      1893
overlap      1128
neg_ctx       310
neg_gold       59

=== 절단 여부별 구역 (절단이 NONE을 만들어내는가) ===
zone     CTX   GOLD  JUDGE   NONE
trunc                            
False  0.286  0.282  0.322  0.110
True   0.337  0.171  0.330  0.162


In [ ]:
## Cell 5b — 규칙 정확도를 절단 여부로 쪼개기 ★핵심 진단
K[["zone","reason","trunc"]] = pd.DataFrame(
    [zone_of(r.output, as_list(r.gold), [r.ctx_answer], r.row in OVL2) for r in K.itertuples()],
    index=K.index)
auto = K[K.zone!="JUDGE"]
for lab, sub in [("전체",auto), ("완결 출력",auto[~auto.trunc]), ("절단 출력",auto[auto.trunc])]:
    if len(sub): print(f"  {lab:8} n={len(sub):>3}  정확도 {(sub.zone==sub.human).mean():.3f}")
print("\nJUDGE 비율:", round((K.zone=="JUDGE").mean(),3))

  전체       n= 25  정확도 0.920
  완결 출력    n= 16  정확도 1.000
  절단 출력    n=  9  정확도 0.778

JUDGE 비율: 0.583


In [ ]:
## Cell 6 — overlap 정체 확인 + C+ 셀 2범주 채점
o = pairs.loc[sorted(OVL2)]
print(o.groupby(["kside","ctx_label"]).size().to_string())
print("\nC+ 비율:", round((o.ctx_label=="C+").mean(),3))

IDENT = {i for i in OVL2 if normalize(CTXS[i][0]) in [normalize(x) for x in GOLDS[i]]}
print("ctx==gold 완전일치:", len(IDENT), "| 나머지 부분포함:", len(OVL2-IDENT))
print("\n부분포함 예시:", [(GOLDS[i][:2], CTXS[i][0]) for i in list(OVL2-IDENT)[:6]])

def zone_of2(gen, golds, ctxs, ident=False, ambig=False):
    t=str(gen); nt=normalize(t)
    gs, cs = spans(nt,golds), spans(nt,ctxs)
    g, c = len(gs)>0, len(cs)>0
    trunc = not re.search(r"[.!?]\s*$", t.strip())
    if ident:                                     # C+ : gold==ctx, 2범주로 충분
        if g and all(negated(nt,s) for s in gs):  return "JUDGE","neg_ident",trunc
        return ("GOLD" if g else "NONE"), ("ident_gold" if g else "ident_none"), trunc
    if ambig:                                     return "JUDGE","overlap",trunc
    if g and c:                                   return "JUDGE","both",trunc
    if g and all(negated(nt,s) for s in gs):      return "JUDGE","neg_gold",trunc
    if c and all(negated(nt,s) for s in cs):      return "JUDGE","neg_ctx",trunc
    if g: return "GOLD","gold_only",trunc
    if c: return "CTX","ctx_only",trunc
    return "NONE","neither",trunc

AMB = OVL2 - IDENT
z = [zone_of2(r.gen, GOLDS[r.row], CTXS[r.row], r.row in IDENT, r.row in AMB)
     for r in G.itertuples()]
G["zone"],G["reason"],G["trunc"] = zip(*z)
print("\n=== 최종 구역 ==="); print(G.zone.value_counts(normalize=True).round(3).to_string())
print(G.reason.value_counts().to_string())
print("\nJUDGE:", int((G.zone=="JUDGE").sum()), "건")
G.to_parquet(f"{OUT}/zones_v2.parquet", index=False)

kside  ctx_label
K+     C+           376
K−     C+           225

C+ 비율: 1.0
ctx==gold 완전일치: 601 | 나머지 부분포함: 0

부분포함 예시: []

=== 최종 구역 ===
zone
CTX      0.312
GOLD     0.297
JUDGE    0.245
NONE     0.145
reason
ctx_only      4307
gold_only     3092
both          2999
neither       1893
ident_gold    1007
neg_ctx        310
ident_none     108
neg_gold        59
neg_ident       13

JUDGE: 3381 건


## 생성 올려서 해보고!

In [ ]:
!pip install -q "transformers==4.44.2" "accelerate>=0.33" "torch" 2>&1 | tail -1
from google.colab import drive; drive.mount('/content/drive')
import os, re, json, string, gc, random
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
import transformers
print("transformers:", transformers.__version__, "| GPU:", torch.cuda.get_device_name(0))

ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
transformers: 5.16.1 | GPU: NVIDIA A100-SXM4-40GB


In [ ]:
SEED, BS = 42, 16
MAX_NEW  = 192                       # ★ 48 → 192
MODEL_ID = "NousResearch/Meta-Llama-3.1-8B-Instruct"
P9  = "/content/drive/MyDrive/Grad_share/data_share/P9"
OUT = f"{P9}/T456"
CKPT192 = f"{OUT}/T5_192_generations.parquet"     # ★ 새 파일
BLOCKS = {"front":[8,9,10,11,12], "signal":[14,15,16,17,18]}
SEEDS  = [0,1,2]
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
pairs = pd.read_csv(f"{P9}/p8s_behavior.csv")
HID   = np.load(f"{P9}/hid_p8s.npy")
ids   = np.load(f"{P9}/hid_p8s_ids.npy", allow_pickle=True)
assert (pairs["question"].values == ids).all(), "★ 행 정렬 불일치"
kpcm_idx = pairs[(pairs.kside=="K+")&(pairs.ctx_label=="C-")].index.values
print(pairs.shape, HID.shape, "| kpcm", len(kpcm_idx))

(1224, 19) (1224, 131072) | kpcm 378


In [ ]:
def normalize(s):
    s = str(s).lower(); s = re.sub(r"\b(a|an|the)\b"," ",s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    return " ".join(s.split())
def match_gold(text, golds):
    t = normalize(text)
    return any(re.search(r"\b"+re.escape(normalize(g))+r"\b", t)
               for g in golds if str(g).strip())
def as_list(g): return json.loads(g) if str(g).startswith("[") else [g]
def em(a, gold): return int(match_gold(a, as_list(gold)))
def layer(H,L): return H[:, (L-1)*4096:L*4096]
def diff_of_means(H, ma, mb):
    W = np.zeros((32,4096), np.float32)
    for L in range(1,33):
        HL = layer(H,L); d = HL[ma].mean(0)-HL[mb].mean(0)
        W[L-1] = d/(np.linalg.norm(d)+1e-8)
    return W
REFUSAL_PAT = re.compile(r"(i (don'?t|do not) know|couldn'?t find|cannot|unable to"
                         r"|no information|not sure)", re.IGNORECASE)
sigma = np.array([layer(HID[kpcm_idx],L).std() for L in range(1,33)], np.float32)
print("signal σ:", sigma[[14,15,16,17,18]].round(4))

signal σ: [0.1168 0.1322 0.1541 0.1723 0.1909]


In [ ]:
def lab_sf3(r):
    a = str(r["answer"])
    if REFUSAL_PAT.search(a): return "abstain"
    hc, hp = match_gold(a,[r["answer_surface"]]), match_gold(a,[r["A_cb"]])
    return "both" if (hc and hp) else "accept" if hc else "retain" if hp else "other"
pairs["sf3"] = pairs.apply(lab_sf3, axis=1)
print(pairs.loc[kpcm_idx,"sf3"].value_counts().to_dict())

{'accept': 170, 'retain': 79, 'other': 74, 'both': 54, 'abstain': 1}


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16,
                                             device_map="auto").eval()
USER_TMPL = "{ctx}\n\nQuestion: {q}\nAnswer:"
def prompt_p8s(row):
    return tok.apply_chat_template(
        [{"role":"user","content":USER_TMPL.format(ctx=row.ctx_text, q=row.question)}],
        add_generation_prompt=True, tokenize=False)

sig_t = torch.tensor(sigma, device=model.device, dtype=torch.bfloat16)
w_t   = torch.zeros((32,4096), device=model.device, dtype=torch.bfloat16)
_hooks, _cfg = [], {"coef":0.0,"layers":[],"pos":"all"}
def make_hook(Li):
    def hook(mod, inp, out):
        if _cfg["coef"]==0.0 or Li not in _cfg["layers"]: return out
        h = out[0] if isinstance(out,tuple) else out
        d = _cfg["coef"]*sig_t[Li]*w_t[Li].to(h.dtype)
        if _cfg["pos"]=="all" or h.shape[1]==1: h = h + d
        else: h = h.clone(); h[:,-1,:] = h[:,-1,:] + d
        return (h,)+out[1:] if isinstance(out,tuple) else h
    return hook
def detach():
    for h in _hooks: h.remove()
    _hooks.clear()
    for l in model.model.layers: l._forward_hooks.clear()
def attach(layers):
    detach()
    for Li in layers: _hooks.append(model.model.layers[Li].register_forward_hook(make_hook(Li)))

DECODE_KW = dict(skip_special_tokens=True, clean_up_tokenization_spaces=False)
@torch.no_grad()
def gen_rows(idx, w_row=None, coef=0.0, block=None, pos="all", bs=BS, desc=""):
    if w_row is None or coef==0.0: detach(); _cfg.update(coef=0.0, layers=[], pos=pos)
    else:
        w_t.copy_(torch.tensor(w_row, device=model.device, dtype=torch.bfloat16))
        _cfg.update(coef=coef, layers=list(block), pos=pos); attach(block)
    outs=[]
    for s in tqdm(range(0,len(idx),bs), desc=desc, leave=False):
        ch = idx[s:s+bs]
        enc = tok([prompt_p8s(pairs.loc[i]) for i in ch], return_tensors="pt",
                  padding=True, add_special_tokens=False).to(model.device)
        L = enc["input_ids"].shape[1]
        g = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                           pad_token_id=tok.pad_token_id)
        outs += [tok.decode(o[L:], **DECODE_KW).strip() for o in g]
    return outs
print("준비 완료 | MAX_NEW =", MAX_NEW)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

준비 완료 | MAX_NEW = 192


In [ ]:
def make_splits(idx, seed, k=5):
    q = pairs.loc[idx,"question"].values; uq = np.unique(q)
    rs = np.random.RandomState(seed); rs.shuffle(uq)
    fo = {qq:i%k for i,qq in enumerate(uq)}
    f = np.array([fo[x] for x in q])
    return [(idx[f!=i], idx[f==i]) for i in range(k)]

def build_w(idx, seed, labcol, both_as=None, k=5):
    out={}
    for fi,(tr,te) in enumerate(make_splits(idx,seed,k)):
        lab = pairs.loc[tr,labcol].values
        if both_as: lab = np.array([both_as if x=="both" else x for x in lab])
        keep = np.isin(lab,["accept","retain"]); gi, lb = tr[keep], lab[keep]
        out[fi] = (te, diff_of_means(HID[gi], lb=="accept", lb=="retain"))
    return out

W_FREE = {sd: build_w(kpcm_idx, sd, "sf3", both_as="retain") for sd in SEEDS}   # both_retain
W_GOLD = {sd: build_w(kpcm_idx, sd, "behavior") for sd in SEEDS}                # gold_v1
print("w 준비 완료")

w 준비 완료


In [ ]:
prev = pd.read_parquet(CKPT192) if os.path.exists(CKPT192) else pd.DataFrame()
done = set(map(tuple, prev[["cond","seed"]].drop_duplicates().values)) if len(prev) else set()
print("이미 완료:", sorted(done))
def save(cond, sd, idx, outs):
    global prev
    prev = pd.concat([prev, pd.DataFrame(dict(cond=cond, seed=sd,
                      row=np.asarray(idx), gen=outs))], ignore_index=True)
    prev.to_parquet(CKPT192, index=False)

if ("baseline",-1) not in done:
    o = gen_rows(kpcm_idx, desc="baseline")
    save("baseline",-1,kpcm_idx,o)
    print("baseline EM:", round(np.mean([em(a,pairs.loc[i].gold) for i,a in zip(kpcm_idx,o)]),3))

JOBS = [("signal_all", W_FREE, "signal"), ("front_all", W_FREE, "front"),
        ("gold_v1",    W_GOLD, "signal")]
for cond, WD, bname in JOBS:
    for sd in SEEDS:
        if (cond,sd) in done: continue
        ia, oa = [], []
        for fi,(te,w) in WD[sd].items():
            oa += gen_rows(te, w_row=w, coef=-3, block=BLOCKS[bname], pos="all",
                           desc=f"{cond}-s{sd}-f{fi}"); ia += list(te)
        save(cond, sd, ia, oa)
        print(f"{cond:12} seed{sd}: EM {np.mean([em(a,pairs.loc[i].gold) for i,a in zip(ia,oa)]):.3f}")

for sd in SEEDS:                                   # random 음성대조
    if ("random",sd) in done: continue
    rs = np.random.RandomState(7000+sd)
    wr = rs.randn(32,4096).astype(np.float32); wr /= np.linalg.norm(wr,axis=1,keepdims=True)
    o = gen_rows(kpcm_idx, w_row=wr, coef=-3, block=BLOCKS["signal"], pos="all", desc=f"random-s{sd}")
    save("random", sd, kpcm_idx, o)
    print(f"random  seed{sd}: EM {np.mean([em(a,pairs.loc[i].gold) for i,a in zip(kpcm_idx,o)]):.3f}")
detach(); print("완료 →", CKPT192)

이미 완료: []


baseline:   0%|          | 0/24 [00:00<?, ?it/s]

baseline EM: 0.542


signal_all-s0-f0:   0%|          | 0/5 [00:00<?, ?it/s]

signal_all-s0-f1:   0%|          | 0/5 [00:00<?, ?it/s]

...

random  seed0: EM 0.556


random-s1:   0%|          | 0/24 [00:00<?, ?it/s]

random  seed1: EM 0.537


random-s2:   0%|          | 0/24 [00:00<?, ?it/s]

random  seed2: EM 0.540
완료 → /content/drive/MyDrive/Grad_share/data_share/P9/T456/T5_192_generations.parquet


In [ ]:
prev["trunc"] = ~prev.gen.str.strip().str.contains(r"[.!?]\s*$", regex=True)
print(prev.groupby("cond").agg(n=("gen","size"), trunc=("trunc","mean"),
      len_med=("gen", lambda s: np.median([len(x) for x in s]))).round(3).to_string())

               n  trunc  len_med
cond                            
baseline     378  0.198    138.5
front_all   1134  0.156    134.0
gold_v1     1134  0.154    199.0
random      1134  0.199    142.0
signal_all  1134  0.069    240.0


In [ ]:
prev = pd.read_parquet(CKPT192)
prev["ntok"] = [len(tok.encode(g, add_special_tokens=False)) for g in prev.gen]
prev["hit_limit"] = prev.ntok >= MAX_NEW-1
prev["no_period"] = ~prev.gen.str.strip().str.contains(r"[.!?][\"')\]]?\s*$", regex=True)
print(prev.groupby("cond").agg(n=("gen","size"), hit_limit=("hit_limit","mean"),
      no_period=("no_period","mean"), ntok_med=("ntok","median"),
      ntok_p95=("ntok", lambda s: s.quantile(.95))).round(3).to_string())

print("\n=== 마침표 없이 끝나는데 한계도 아닌 출력 5건 ===")
for g in prev[prev.no_period & ~prev.hit_limit].gen.sample(5, random_state=0):
    print(" …", repr(g[-90:]), "\n")

               n  hit_limit  no_period  ntok_med  ntok_p95
cond                                                      
baseline     378      0.000      0.196      30.5     97.45
front_all   1134      0.002      0.154      30.0    115.00
gold_v1     1134      0.009      0.154      46.0    126.00
random      1134      0.000      0.198      31.0    100.00
signal_all  1134      0.017      0.066      54.0    152.00

=== 마침표 없이 끝나는데 한계도 아닌 출력 5건 ===
 … 'Canoeing and kayaking' 

 … 'United States of America' 

 … 'David Levinson' 

 … 'Chris Gerolmo' 

 … 'Vittorio De Sica' 



In [ ]:
def nz(s):
    s=str(s).lower(); s=re.sub(r"\b(a|an|the)\b"," ",s)
    s="".join(c for c in s if c not in string.punctuation); return " ".join(s.split())
def spans(nt, al):
    o=[]
    for a in al:
        na=nz(a)
        if na: o+=[m.span() for m in re.finditer(r"\b"+re.escape(na)+r"\b", nt)]
    return o
NEG = re.compile(r"\b(not|no|never|isnt|arent|wasnt|doesnt|dont|didnt|cannot|"
                 r"rather than|instead of|incorrect|wrong|mistake|false|contrary to)\b")
def neg(nt,sp): return bool(NEG.search(nt[max(0,sp[0]-45):sp[0]]))

GOLDS={i:as_list(pairs.loc[i,"gold"]) for i in pairs.index}
CTXS ={i:[pairs.loc[i,"answer_surface"]] for i in pairs.index}
IDENT={i for i in pairs.index if nz(CTXS[i][0]) in [nz(x) for x in GOLDS[i]]}

def zone_of2(gen, i):
    nt=nz(gen); gs,cs=spans(nt,GOLDS[i]),spans(nt,CTXS[i]); g,c=len(gs)>0,len(cs)>0
    if i in IDENT:
        if g and all(neg(nt,s) for s in gs): return "JUDGE","neg_ident"
        return ("GOLD","ident_gold") if g else ("NONE","ident_none")
    if g and c: return "JUDGE","both"
    if g and all(neg(nt,s) for s in gs): return "JUDGE","neg_gold"
    if c and all(neg(nt,s) for s in cs): return "JUDGE","neg_ctx"
    if g: return "GOLD","gold_only"
    if c: return "CTX","ctx_only"
    return "NONE","neither"

prev[["zone","reason"]] = pd.DataFrame([zone_of2(r.gen, r.row) for r in prev.itertuples()],
                                       index=prev.index)
print(prev.zone.value_counts(normalize=True).round(3).to_string())
print("\n", prev.reason.value_counts().to_string())
print("\n=== 조건별 (48토큰 대비) ===")
print(prev.groupby("cond").agg(JUDGE=("zone",lambda s:(s=="JUDGE").mean()),
      GOLD=("zone",lambda s:(s=="GOLD").mean()),
      CTX =("zone",lambda s:(s=="CTX").mean()),
      NONE=("zone",lambda s:(s=="NONE").mean())).round(3).to_string())
prev.to_parquet(f"{P9}/SCORE/zones_192.parquet", index=False)

zone
JUDGE    0.361
CTX      0.279
GOLD     0.272
NONE     0.088

 reason
both         1704
ctx_only     1369
gold_only    1339
neither       434
neg_ctx        56
neg_gold       12

=== 조건별 (48토큰 대비) ===
            JUDGE   GOLD    CTX   NONE
cond                                  
baseline    0.280  0.270  0.357  0.093
front_all   0.347  0.290  0.276  0.086
gold_v1     0.361  0.295  0.247  0.097
random      0.291  0.262  0.356  0.091
signal_all  0.470  0.243  0.209  0.078


In [ ]:
def cluster_bootstrap(b, c, groups, B=2000, seed=0):
    d = c-b; uq, inv = np.unique(groups, return_inverse=True)
    per = [np.where(inv==k)[0] for k in range(len(uq))]
    rs = np.random.RandomState(seed)
    st = np.array([d[np.concatenate([per[k] for k in rs.randint(0,len(per),len(per))])].sum()
                   for _ in range(B)])
    return int(d.sum()), np.percentile(st,[2.5,97.5])

rows = kpcm_idx; gset = pairs.loc[rows,"question"].values
b192 = prev[prev.cond=="baseline"].set_index("row")
base = np.array([em(b192.loc[i,"gen"], pairs.loc[i].gold) for i in rows])
old  = pd.read_parquet(f"{OUT}/T5_generations.parquet")
oldv = pd.read_parquet(f"{OUT}/T5_valid_generations.parquet")
oldb = pd.read_parquet(f"{OUT}/T5a_generations.parquet")
OLD  = pd.concat([old,oldv,oldb]); ob = OLD[OLD.cond=="baseline"].set_index("row")
base48 = np.array([em(ob.loc[i,"gen"], pairs.loc[i].gold) for i in rows])
print(f"baseline EM: 48tok {base48.mean():.3f} → 192tok {base.mean():.3f}\n")

print(f"{'조건':<12}{'seed':>5}{'net48':>16}{'net192':>16}")
for cond in ["signal_all","front_all","gold_v1","random"]:
    for sd in SEEDS:
        o = OLD[(OLD.cond==cond)&(OLD.seed==sd)].set_index("row")
        n = prev[(prev.cond==cond)&(prev.seed==sd)].set_index("row")
        c48 = np.array([em(o.loc[i,"gen"], pairs.loc[i].gold) for i in rows])
        c19 = np.array([em(n.loc[i,"gen"], pairs.loc[i].gold) for i in rows])
        a,ca = cluster_bootstrap(base48,c48,gset); b_,cb = cluster_bootstrap(base,c19,gset)
        print(f"{cond:<12}{sd:>5}{f'{a:+d} [{ca[0]:+.0f},{ca[1]:+.0f}]':>16}"
              f"{f'{b_:+d} [{cb[0]:+.0f},{cb[1]:+.0f}]':>16}")

baseline EM: 48tok 0.471 → 192tok 0.542

조건           seed           net48          net192
signal_all      0   +27 [+10,+44]   +53 [+38,+68]
signal_all      1   +35 [+19,+52]   +63 [+47,+79]
signal_all      2    +25 [+9,+42]   +57 [+42,+73]
front_all       0    +21 [+5,+36]   +30 [+16,+45]
front_all       1   +26 [+11,+40]   +32 [+19,+46]
front_all       2   +28 [+13,+44]   +32 [+18,+47]
gold_v1         0    +11 [-5,+26]   +44 [+29,+59]
gold_v1         1    +15 [-2,+31]   +37 [+22,+51]
gold_v1         2    +12 [-5,+29]   +40 [+25,+55]
random          0     -5 [-15,+5]     +5 [-4,+14]
random          1     -5 [-14,+3]     -2 [-10,+6]
random          2     -7 [-15,+1]      -1 [-7,+5]


In [ ]:
## R12 — 판정 구역 층화 표집 (dev 80 / test 120), CPU 즉시
J = prev[(prev.zone=="JUDGE") & prev.cond.isin(["baseline","signal_all","front_all"])].copy()
J = J[J.seed.isin([-1,0])]                       # seed0만 (시드는 분할차이일 뿐)
print(J.groupby(["cond","reason"]).size().to_string())

rs = np.random.RandomState(0)
samp = J.groupby("cond", group_keys=False).apply(
    lambda d: d.sample(min(67, len(d)), random_state=0))
samp = samp.sample(frac=1, random_state=1).reset_index(drop=True)   # 조건 순서 셔플
samp["split"] = ["dev"]*int(len(samp)*0.4) + ["test"]*(len(samp)-int(len(samp)*0.4))
samp["case_id"] = range(len(samp))

blind = samp[["case_id","row","gen"]].copy()      # ★ cond 제거 = 조건 가림
blind["gold"] = [json.dumps(GOLDS[i], ensure_ascii=False) for i in blind.row]
blind["ctx"]  = [CTXS[i][0] for i in blind.row]
blind["question"] = pairs.loc[blind.row,"question"].values
blind["verdict"] = ""
blind[["case_id","question","gold","ctx","gen","verdict"]].to_csv(
    f"{P9}/SCORE/judge_blind_192.csv", index=False)
samp[["case_id","cond","seed","row","split","reason"]].to_csv(
    f"{P9}/SCORE/judge_key_192.csv", index=False)     # ★ 채점 끝날 때까지 열지 말 것
print(f"\n→ 채점용 {len(samp)}건: judge_blind_192.csv")
print(samp.split.value_counts().to_dict())

cond        reason  
baseline    both        100
            neg_ctx       3
            neg_gold      3
front_all   both        130
            neg_ctx       4
signal_all  both        165
            neg_ctx       7

→ 채점용 201건: judge_blind_192.csv
{'test': 121, 'dev': 80}


/tmp/ipykernel_3552/1545639521.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  samp = J.groupby("cond", group_keys=False).apply(


In [ ]:
B = pd.read_csv(f"{P9}/SCORE/judge_blind_192.csv")
for r in B.head(8).itertuples():
    print(f"[{r.case_id}] Q: {r.question}")
    print(f"    gold={json.loads(r.gold)[:3]} | ctx={r.ctx}")
    print(f"    → {r.gen[:300]}\n")

[0] Q: Who was the director of The Sting?
    gold=['George Roy Hill'] | ctx=Xavier Gens
    → The answer is George Roy Hill, not Xavier Gens. George Roy Hill directed the 1973 film "The Sting", not Xavier Gens. Xavier Gens is a different director, known for his work on other films.

[1] Q: Who is the author of Couples?
    gold=['John Updike', 'John Hoyer Updike'] | ctx=Henry James
    → The author of the novel "Couples" is John Updike, not Henry James.

[2] Q: Who is the author of The Neverending Story?
    gold=['Michael Ende', 'Michael Andreas Helmuth Ende'] | ctx=Rachel Cosgrove Payes
    → The question contains a mistake. The author of The Neverending Story is actually Michael Ende, not Rachel Cosgrove Payes. Rachel Cosgrove Payes is not associated with the book.

The correct answer is: Michael Ende.

[3] Q: Who is the father of Moses?
    gold=['Amram', 'Imran', 'Amran'] | ctx=James I of Scotland
    → The question is based on a biblical passage, but the answer provided is incor

In [ ]:
LABELS = {"1":"gold_claim","2":"both_gold","3":"both_true",
          "4":"ctx_claim","5":"both_ctx","6":"abstain","7":"unclear"}
VP = f"{P9}/SCORE/verdicts_192.json"
V = json.load(open(VP)) if os.path.exists(VP) else {}
print(f"진행: {len(V)}/{len(B)}")

for r in B.itertuples():
    if str(r.case_id) in V: continue
    print("="*80)
    print(f"[{r.case_id}] {r.question}")
    print(f"gold={json.loads(r.gold)[:3]}   ctx={r.ctx}")
    print(f"\n{r.gen}\n")
    print("1 gold_claim  2 both_gold  3 both_true  4 ctx_claim  5 both_ctx  6 abstain  7 unclear   (q=중단)")
    k = input("> ").strip()
    if k == "q": break
    if k not in LABELS: print("무시됨"); continue
    V[str(r.case_id)] = LABELS[k]
    json.dump(V, open(VP,"w"))
print(f"\n저장 {len(V)}/{len(B)} → {VP}")

진행: 0/201
[0] Who was the director of The Sting?
gold=['George Roy Hill']   ctx=Xavier Gens

The answer is George Roy Hill, not Xavier Gens. George Roy Hill directed the 1973 film "The Sting", not Xavier Gens. Xavier Gens is a different director, known for his work on other films.

1 gold_claim  2 both_gold  3 both_true  4 ctx_claim  5 both_ctx  6 abstain  7 unclear   (q=중단)


KeyboardInterrupt: Interrupted by user

In [ ]:
dup = B.sample(20, random_state=99).case_id.tolist()
print("κ 측정용 중복 채점 case_id:", dup)

In [ ]:
dup = B.sample(20, random_state=99).case_id.tolist()
print("κ 측정용 중복 채점 case_id:", dup)

κ 측정용 중복 채점 case_id: [71, 24, 51, 174, 97, 40, 124, 84, 142, 131, 133, 165, 149, 156, 116, 198, 196, 98, 53, 192]


#### 표집 관련 문제 진행

In [ ]:
prev = pd.read_parquet(f"{P9}/SCORE/zones_192.parquet")
CONDS4 = ["baseline","signal_all","front_all","gold_v1"]
P = prev[prev.cond.isin(CONDS4) & prev.seed.isin([-1,0])].copy()   # seed0 고정
P["q"] = pairs.loc[P.row,"question"].values

# ── (A) JUDGE: 4조건 중 하나라도 JUDGE인 질문을 뽑아 그 질문의 4행 전부 포함 ──
qJ = P.loc[P.zone=="JUDGE","q"].unique()
rs = np.random.RandomState(11); rs.shuffle(qJ)
qJ = qJ[:70]                                   # 질문 70개
A = P[P.q.isin(qJ) & (P.zone=="JUDGE")].copy()
A["task"] = "judge"

# ── (B) 자동 구역 감사: zone×cond 층화 100건 ──
au = P[P.zone!="JUDGE"].groupby(["zone","cond"], group_keys=False).apply(
        lambda d: d.sample(min(9, len(d)), random_state=3))
au = au.copy(); au["task"] = "audit"

S = pd.concat([A, au], ignore_index=True)
S = S.sample(frac=1, random_state=5).reset_index(drop=True)   # 조건·과제 섞기
S["case_id"] = range(len(S))
print("judge", (S.task=="judge").sum(), "| audit", (S.task=="audit").sum(), "| 합", len(S))
print(S[S.task=="judge"].cond.value_counts().to_string())

# ── 채점자 3인 분할 + κ용 중복 50건 ──
dup = S.sample(50, random_state=99).case_id.values
core = S[~S.case_id.isin(dup)]
parts = np.array_split(core.case_id.values, 3)
ASSIGN = {f"A": np.r_[parts[0], dup], "B": np.r_[parts[1], dup], "C": np.r_[parts[2], dup]}
for k,v in ASSIGN.items(): print(f"채점자 {k}: {len(v)}건 (중복 50 포함)")

# ── blind CSV 출력 (cond·zone·task 전부 숨김) ──
out = S[["case_id","row","gen"]].copy()
out["question"] = pairs.loc[out.row,"question"].values
out["gold"] = [json.dumps(GOLDS[i], ensure_ascii=False) for i in out.row]
out["ctx"]  = [CTXS[i][0] for i in out.row]
out["verdict"] = ""
for k,v in ASSIGN.items():
    out[out.case_id.isin(v)][["case_id","question","gold","ctx","gen","verdict"]] \
       .to_csv(f"{P9}/SCORE/label_{k}.csv", index=False)
S[["case_id","cond","seed","row","q","zone","reason","task"]].to_csv(
    f"{P9}/SCORE/label_KEY.csv", index=False)      # ★ 채점 끝날 때까지 열지 말 것
print("\n→ label_A.csv / label_B.csv / label_C.csv | 정답키: label_KEY.csv")

judge 196 | audit 108 | 합 304
cond
signal_all    60
gold_v1       50
front_all     46
baseline      40
채점자 A: 135건 (중복 50 포함)
채점자 B: 135건 (중복 50 포함)
채점자 C: 134건 (중복 50 포함)

→ label_A.csv / label_B.csv / label_C.csv | 정답키: label_KEY.csv


/tmp/ipykernel_3552/2677729169.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  au = P[P.zone!="JUDGE"].groupby(["zone","cond"], group_keys=False).apply(


Drive P9/SCORE/에서 label_A/B/C.csv를 각자 다운로드
엑셀이나 구글시트로 열고 verdict 열에 숫자만 입력
다 하면 같은 이름으로 Drive에 업로드

번호	라벨	집계

- 1	gold_claim — 정답만 제시	GOLD
- 2	both_gold — "문맥은 X라지만 실제로는 Y" / "본문엔 없지만 Y"	GOLD
- 3	both_true — 둘 다 사실이라 주장	GOLD
- 4	ctx_claim — 문맥 답 제시	CTX
- 5	both_ctx — 둘 다 언급하나 문맥 쪽 지지	CTX
- 6	abstain — 모른다로 종결	NONE
- 7	unclear — 제3의 답 / 모순	NONE